# 00 Investigate Flow Duplicate Records

This notebook investigates duplicate `(station_id, timestamp)` records in `final_hourly_flow_allfeature.csv`.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("data/nsw2025")
NEW_DIR = ROOT / "new data"

OUT_DIR = ROOT / "data_quality_checks"
OUT_DIR.mkdir(exist_ok=True)

FLOW_FILE = NEW_DIR / "final_hourly_flow_allfeature.csv"

print("Flow file:", FLOW_FILE)
print("Output folder:", OUT_DIR)

Flow file: data\nsw2025\new data\final_hourly_flow_allfeature.csv
Output folder: data\nsw2025\data_quality_checks


## 2. Load final hourly flow dataset

In [2]:
flow_all = pd.read_csv(
    FLOW_FILE,
    dtype={
        "station_id": str,
        "incident_id": str
    },
    low_memory=False
)

flow_all["station_id"] = flow_all["station_id"].astype(str).str.strip()

flow_all["timestamp"] = pd.to_datetime(
    flow_all["timestamp"],
    format="mixed",
    errors="coerce"
)

print("Shape:", flow_all.shape)
print("Missing timestamps:", flow_all["timestamp"].isna().sum())
print("Unique stations:", flow_all["station_id"].nunique())
print("Timestamp range:", flow_all["timestamp"].min(), "to", flow_all["timestamp"].max())

flow_all.head()

Shape: (468530, 10)
Missing timestamps: 0
Unique stations: 67
Timestamp range: 2025-01-01 00:00:00 to 2025-12-31 23:00:00


,station_id,timestamp,total_flow,lane_count,road_functional_hierarchy,distance_to_intersection,incident_id,is_major_incident,impact_sequence_hour,precipitation
0,100001,2025-01-01 00:00:00,201.0,TwoOrMoreLanes,Primary Road,80,NO_INCIDENT,0.0,-1.0,0.0
1,100001,2025-01-01 01:00:00,248.0,TwoOrMoreLanes,Primary Road,80,NO_INCIDENT,0.0,-1.0,0.0
2,100001,2025-01-01 02:00:00,171.0,TwoOrMoreLanes,Primary Road,80,NO_INCIDENT,0.0,-1.0,0.0
3,100001,2025-01-01 03:00:00,144.0,TwoOrMoreLanes,Primary Road,80,NO_INCIDENT,0.0,-1.0,0.0
4,100001,2025-01-01 04:00:00,81.0,TwoOrMoreLanes,Primary Road,80,NO_INCIDENT,0.0,-1.0,0.0


## 3. Check duplicate station-hour records

A valid traffic-flow time series should contain one `total_flow` value for each `(station_id, timestamp)` pair. This section counts duplicated station-hour keys.

In [3]:
key_cols = ["station_id", "timestamp"]

station_hour_counts = (
    flow_all
    .groupby(key_cols)
    .size()
    .reset_index(name="row_count")
)

duplicate_keys = station_hour_counts[
    station_hour_counts["row_count"] > 1
].copy()

print("Total station-hour keys:", len(station_hour_counts))
print("Duplicate station-hour keys:", len(duplicate_keys))

duplicate_keys["row_count"].value_counts().sort_index()

Total station-hour keys: 454320
Duplicate station-hour keys: 5178


row_count
2     2700
3      784
4      595
5      232
6      326
7      102
8       59
9       50
10      24
11     140
12      61
13      14
14      11
15       9
16      15
17       6
18       7
19       5
20       6
21       5
22       2
23       4
24       5
25       2
26       2
27       1
28       2
29       1
30       1
31       2
32       1
33       1
35       2
36       1
Name: count, dtype: int64

## 4. Extract duplicate rows

This section extracts all rows belonging to duplicated `(station_id, timestamp)` keys.

In [4]:
duplicate_rows = flow_all.merge(
    duplicate_keys[key_cols + ["row_count"]],
    on=key_cols,
    how="inner"
).sort_values(key_cols + ["total_flow"])

print("Duplicate rows:", len(duplicate_rows))

duplicate_rows.head(20)

Duplicate rows: 19388


,station_id,timestamp,total_flow,lane_count,road_functional_hierarchy,distance_to_intersection,incident_id,is_major_incident,impact_sequence_hour,precipitation,row_count
0,100001,2025-06-27 00:00:00,414.0,TwoOrMoreLanes,Primary Road,80,240340.0-webtirf,0.0,0.0,5.10,4
1,100001,2025-06-27 00:00:00,414.0,TwoOrMoreLanes,Primary Road,80,240340.0-webtirf,0.0,0.0,5.10,4
2,100001,2025-06-27 00:00:00,414.0,TwoOrMoreLanes,Primary Road,80,240340.0-webtirf,0.0,0.0,5.10,4
3,100001,2025-06-27 00:00:00,414.0,TwoOrMoreLanes,Primary Road,80,240340.0-webtirf,0.0,0.0,5.10,4
4,100001,2025-08-29 05:00:00,1788.0,TwoOrMoreLanes,Primary Road,80,247639-webtirf,0.0,0.0,8.75,4
5,100001,2025-08-29 05:00:00,1788.0,TwoOrMoreLanes,Primary Road,80,247639-webtirf,0.0,0.0,8.75,4
6,100001,2025-08-29 05:00:00,1788.0,TwoOrMoreLanes,Primary Road,80,247639-webtirf,0.0,0.0,8.75,4
7,100001,2025-08-29 05:00:00,1788.0,TwoOrMoreLanes,Primary Road,80,247639-webtirf,0.0,0.0,8.75,4
8,100001,2025-09-27 13:00:00,2628.0,TwoOrMoreLanes,Primary Road,80,250888-webtirf,0.0,0.0,30.20,5
9,100001,2025-09-27 13:00:00,2628.0,TwoOrMoreLanes,Primary Road,80,250888-webtirf,0.0,0.0,30.20,5


## 5. Separate harmless duplicates from conflicting flow values

Some duplicates may be exact repeated rows or repeated incident-feature rows with the same traffic flow. The main issue occurs when the same `(station_id, timestamp)` has more than one distinct `total_flow` value.

In [5]:
duplicate_summary = (
    duplicate_rows
    .groupby(key_cols)
    .agg(
        row_count=("total_flow", "size"),
        n_unique_flow=("total_flow", "nunique"),
        min_flow=("total_flow", "min"),
        max_flow=("total_flow", "max"),
        flow_values=("total_flow", lambda x: sorted(x.dropna().unique().tolist())),
        n_unique_incident_id=("incident_id", "nunique"),
        incident_ids=("incident_id", lambda x: sorted(x.dropna().unique().tolist())[:10])
    )
    .reset_index()
)

conflicting_flow_keys = duplicate_summary[
    duplicate_summary["n_unique_flow"] > 1
].copy()

same_flow_duplicate_keys = duplicate_summary[
    duplicate_summary["n_unique_flow"] == 1
].copy()

print("Duplicate keys with same flow:", len(same_flow_duplicate_keys))
print("Duplicate keys with conflicting flow:", len(conflicting_flow_keys))

conflicting_flow_keys.head(50)

Duplicate keys with same flow: 5138
Duplicate keys with conflicting flow: 40


,station_id,timestamp,row_count,n_unique_flow,min_flow,max_flow,flow_values,n_unique_incident_id,incident_ids
3,100001,2025-10-21,2,2,184.0,240.0,"[184.0, 240.0]",1,[NO_INCIDENT]
80,23067,2025-10-21,2,2,20.0,201.0,"[20.0, 201.0]",1,[NO_INCIDENT]
113,29005,2025-10-21,2,2,58.0,669.0,"[58.0, 669.0]",1,[NO_INCIDENT]
212,34016,2025-10-21,2,2,0.0,50.0,"[0.0, 50.0]",1,[NO_INCIDENT]
281,47024,2025-10-21,2,2,39.0,800.0,"[39.0, 800.0]",1,[NO_INCIDENT]
414,50260,2025-10-21,2,2,40.0,339.0,"[40.0, 339.0]",1,[NO_INCIDENT]
523,53003,2025-10-21,2,2,36.0,127.0,"[36.0, 127.0]",1,[NO_INCIDENT]
580,53004,2025-10-21,2,2,60.0,126.0,"[60.0, 126.0]",1,[NO_INCIDENT]
610,55049,2025-10-21,2,2,14.0,47.0,"[14.0, 47.0]",1,[NO_INCIDENT]
647,57025,2025-10-21,24,2,2.0,12.0,"[2.0, 12.0]",2,"[253250-webtirf, 253250.0-webtirf]"


## 6. Inspect conflicting flow rows

This section displays all rows where the same station-hour has different `total_flow` values.

In [6]:
conflicting_flow_rows = flow_all.merge(
    conflicting_flow_keys[key_cols],
    on=key_cols,
    how="inner"
).sort_values(key_cols + ["total_flow"])

print("Conflicting flow rows:", len(conflicting_flow_rows))

conflicting_flow_rows.head(100)

Conflicting flow rows: 102


,station_id,timestamp,total_flow,lane_count,road_functional_hierarchy,distance_to_intersection,incident_id,is_major_incident,impact_sequence_hour,precipitation
1,100001,2025-10-21,184.0,TwoOrMoreLanes,Primary Road,80,NO_INCIDENT,0.0,-1.0,0.0
0,100001,2025-10-21,240.0,TwoOrMoreLanes,Primary Road,80,NO_INCIDENT,0.0,-1.0,0.0
3,23067,2025-10-21,20.0,TwoOrMoreLanes,Local Road,100,NO_INCIDENT,0.0,-1.0,0.0
2,23067,2025-10-21,201.0,TwoOrMoreLanes,Local Road,100,NO_INCIDENT,0.0,-1.0,0.0
5,29005,2025-10-21,58.0,TwoOrMoreLanes,Arterial Road,20,NO_INCIDENT,0.0,-1.0,0.0
...,...,...,...,...,...,...,...,...,...,...
94,T0337,2025-10-21,6.0,TwoOrMoreLanes,Arterial Road,210,NO_INCIDENT,0.0,-1.0,0.0
97,T0342,2025-10-21,1.0,TwoOrMoreLanes,Arterial Road,70,NO_INCIDENT,0.0,-1.0,0.0
96,T0342,2025-10-21,16.0,TwoOrMoreLanes,Arterial Road,70,NO_INCIDENT,0.0,-1.0,0.0
98,T0345,2025-10-21,18.0,OneLane,Arterial Road,80,NO_INCIDENT,0.0,-1.0,0.0


## 7. Check which columns vary in conflicting rows

If only `total_flow` varies while all other columns are identical, the issue is likely a flow-processing problem rather than an incident-feature merge issue.

In [7]:
variation_check = (
    conflicting_flow_rows
    .groupby(key_cols)
    .nunique(dropna=False)
)

# Count how many conflict groups have variation in each column
varying_columns = (
    (variation_check > 1)
    .sum()
    .sort_values(ascending=False)
)

varying_columns

total_flow                   40
incident_id                   1
road_functional_hierarchy     0
lane_count                    0
distance_to_intersection      0
is_major_incident             0
impact_sequence_hour          0
precipitation                 0
dtype: int64

## 8. Check whether conflicts occur on specific dates

This helps determine whether the conflicting values are isolated to a specific timestamp or represent a broader dataset issue.

In [8]:
conflicting_flow_keys["date"] = conflicting_flow_keys["timestamp"].dt.date

conflict_by_date = (
    conflicting_flow_keys
    .groupby("date")
    .size()
    .reset_index(name="conflicting_station_hours")
    .sort_values("date")
)

conflict_by_date

,date,conflicting_station_hours
0,2025-10-21,40


## 9. Export data quality reports

The following CSV files are exported for review:

- `duplicate_station_hour_keys.csv`
- `duplicate_station_hour_rows.csv`
- `duplicate_station_hour_summary.csv`
- `conflicting_total_flow_keys.csv`
- `conflicting_total_flow_rows.csv`
- `conflicting_total_flow_by_date.csv`
- `conflicting_column_variation.csv`

In [9]:
duplicate_keys.to_csv(
    OUT_DIR / "duplicate_station_hour_keys.csv",
    index=False
)

duplicate_rows.to_csv(
    OUT_DIR / "duplicate_station_hour_rows.csv",
    index=False
)

duplicate_summary.to_csv(
    OUT_DIR / "duplicate_station_hour_summary.csv",
    index=False
)

conflicting_flow_keys.to_csv(
    OUT_DIR / "conflicting_total_flow_keys.csv",
    index=False
)

conflicting_flow_rows.to_csv(
    OUT_DIR / "conflicting_total_flow_rows.csv",
    index=False
)

conflict_by_date.to_csv(
    OUT_DIR / "conflicting_total_flow_by_date.csv",
    index=False
)

varying_columns.reset_index().rename(
    columns={
        "index": "column",
        0: "number_of_conflict_groups_with_variation"
    }
).to_csv(
    OUT_DIR / "conflicting_column_variation.csv",
    index=False
)

print("Saved reports to:", OUT_DIR)

Saved reports to: data\nsw2025\data_quality_checks


## 10. Summary for data-processing review

In [10]:
print("========== FLOW DUPLICATE INVESTIGATION SUMMARY ==========")

print(f"Total rows: {len(flow_all):,}")
print(f"Unique station-hour keys: {len(station_hour_counts):,}")
print(f"Duplicate station-hour keys: {len(duplicate_keys):,}")
print(f"Rows belonging to duplicated station-hours: {len(duplicate_rows):,}")

print()
print(f"Duplicate station-hours with same flow: {len(same_flow_duplicate_keys):,}")
print(f"Duplicate station-hours with conflicting flow: {len(conflicting_flow_keys):,}")

print()
print("Conflict dates:")
print(conflict_by_date.to_string(index=False))

print()
print("Columns varying in conflicting rows:")
print(varying_columns[varying_columns > 0].to_string())

========== FLOW DUPLICATE INVESTIGATION SUMMARY ==========
Total rows: 468,530
Unique station-hour keys: 454,320
Duplicate station-hour keys: 5,178
Rows belonging to duplicated station-hours: 19,388

Duplicate station-hours with same flow: 5,138
Duplicate station-hours with conflicting flow: 40

Conflict dates:
      date  conflicting_station_hours
2025-10-21                         40

Columns varying in conflicting rows:
total_flow     40
incident_id     1
